# Week 10 — build a tree, score it, and say how sure we are

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top.

The code already runs. Read the comments, change one input if you want, watch the printout. Then answer the questions.

Fitch and pruning on the tree with leaves $\mathtt{C}, \mathtt{C}, \mathtt{T}$; neighbour-joining on four samples; then the four samples' twenty sites: parsimony on all three trees, the bootstrap, and a random walk over trees.

In [ ]:
def fitch_node(R, S):
    inter = R & S
    if inter:
        return inter, 0
    return R | S, 1


print("two C's", fitch_node({"C"}, {"C"}))
print("C with T", fitch_node({"C"}, {"T"}))

left, cost_left = fitch_node({"C"}, {"C"})
root, cost_root = fitch_node(left, {"T"})
print("root set", root, "total cost", cost_left + cost_root)


## Pruning

Same tree. Mix each child with $P$, then multiply. Node $u$ (two `C`s) gets $(0.64, 0.04)$; the root $(0.104, 0.128)$; with both letters equally likely at the root, $L=0.116$.

In [ ]:
STATES = ("C", "T")
P = {("C", "C"): 0.8, ("C", "T"): 0.2, ("T", "C"): 0.2, ("T", "T"): 0.8}


def leaf(letter):
    return {s: 1.0 if s == letter else 0.0 for s in STATES}


def prune_node(L_left, L_right):
    out = {}
    for a in STATES:
        left = sum(P[(a, x)] * L_left[x] for x in STATES)
        right = sum(P[(a, y)] * L_right[y] for y in STATES)
        out[a] = left * right
    return out


u = prune_node(leaf("C"), leaf("C"))
print("parent of C, C:", u)
root = prune_node(u, leaf("T"))
print("root vs T:    ", {k: round(v, 3) for k, v in root.items()})
L = 0.5 * root["C"] + 0.5 * root["T"]
print("likelihood:   ", round(L, 3))
# Play: set P change to 0.4 (longer branch) and re-run.


## Neighbour-joining

Row sums $R$, then $Q_{ij} = (n-2)\,d_{ij} - R_i - R_j$. Join the smallest pair, compute its branch lengths and the distances from the new node.

In [ ]:
names = ["A", "B", "C", "D"]
D = {
    ("A", "B"): 5, ("A", "C"): 9, ("A", "D"): 9,
    ("B", "C"): 10, ("B", "D"): 10, ("C", "D"): 8,
}


def d(a, b):
    return 0 if a == b else D.get((a, b), D.get((b, a)))


n = len(names)
R = {a: sum(d(a, b) for b in names) for a in names}
print("row sums R:", R)

Q = {}
for i, a in enumerate(names):
    for b in names[i + 1:]:
        Q[(a, b)] = (n - 2) * d(a, b) - R[a] - R[b]
        print(f"Q_{a}{b} = {Q[(a, b)]}")

a, b = min(Q, key=Q.get)               # first smallest pair: A, B
da = d(a, b) / 2 + (R[a] - R[b]) / (2 * (n - 2))
print(f"\njoin {a} and {b}: branch {a} = {da}, branch {b} = {d(a, b) - da}")
rest = [x for x in names if x not in (a, b)]
du = {x: (d(a, x) + d(b, x) - d(a, b)) / 2 for x in rest}
print("distances from u:", du)

# three nodes left: u, C, D meet in one centre
x, y = rest
bu = (du[x] + du[y] - d(x, y)) / 2
print(f"last join: u = {bu}, {x} = {du[x] - bu}, {y} = {du[y] - bu}")

## The toy alignment

Four samples, twenty sites. The distances below give exactly the table of the slides: $d_{AB} = 5$, $d_{AC} = 9$, and so on.

In [ ]:
seqs = {
    "A": "ATACTGTAGATCCATGGCGT",
    "B": "GTGTTGTAGATCCATGACGC",
    "C": "ACGTTGCAAGTTCGTAGCGT",
    "D": "ACGCTGCGAGCCCACGACGT",
}
m = len(seqs["A"])


def dist(a, b, cols=range(m)):
    return sum(seqs[a][i] != seqs[b][i] for i in cols)


for a, b in [("A", "B"), ("A", "C"), ("A", "D"), ("B", "C"), ("B", "D"), ("C", "D")]:
    print(a + b, dist(a, b))

## Parsimony on the three trees

A tree on four leaves is a split: two pairs. Run Fitch on every column (the function `fitch_node` from the top) and add up. The slides give 17, 20, 20.

In [ ]:
splits = [("AB", "CD"), ("AC", "BD"), ("AD", "BC")]


def parsimony(split):
    (p, q), (r, s) = split
    total = 0
    for i in range(m):
        left, c1 = fitch_node({seqs[p][i]}, {seqs[q][i]})
        right, c2 = fitch_node({seqs[r][i]}, {seqs[s][i]})
        _, c3 = fitch_node(left, right)
        total += c1 + c2 + c3
    return total


for sp in splits:
    print(sp[0], "|", sp[1], parsimony(sp), "changes")

## The bootstrap

Draw 20 columns with replacement, rebuild the tree, count. On four samples neighbour-joining picks the split with the smallest $d_{xy} + d_{zw}$ (the same choice as the $Q$ rule). Ties get split votes. The slides give 87 %, 7 %, 6 %.

In [ ]:
import random

random.seed(1)


def nj4(cols):
    sums = [dist(x[0], x[1], cols) + dist(y[0], y[1], cols) for x, y in splits]
    best = min(sums)
    return [k for k, v in enumerate(sums) if v == best]


B = 1000
votes = [0.0, 0.0, 0.0]
for _ in range(B):
    cols = [random.randrange(m) for _ in range(m)]   # with replacement
    winners = nj4(cols)
    for k in winners:
        votes[k] += 1 / len(winners)

for sp, v in zip(splits, votes):
    print(sp[0], "|", sp[1], f"{100 * v / B:.0f} %")

## A random walk over the three trees

The log-likelihoods of the three trees (from the slides, branch lengths already optimised). Propose another tree; move if it is better, otherwise move with probability new / current. Count where the walk spends its time, and compare with the posterior $L_i / \sum L$.

In [ ]:
import math

loglik = [-88.17, -90.78, -90.78]
rel = [math.exp(x - max(loglik)) for x in loglik]
posterior = [r / sum(rel) for r in rel]

random.seed(7)
cur, visits, N = 1, [0, 0, 0], 10000
for _ in range(N):
    prop = random.choice([k for k in range(3) if k != cur])
    if random.random() < min(1, math.exp(loglik[prop] - loglik[cur])):
        cur = prop
    visits[cur] += 1

for sp, v, p in zip(splits, visits, posterior):
    print(sp[0], "|", sp[1], f"walk {v / N:.3f}   posterior {p:.3f}")

## Questions

Without looking at notes. **Q1–Q2:** today’s idea. **Q3:** a situation we did not practise.

**Q1.** Why can we not try every tree on 20 samples? What does neighbour-joining do instead, and why does it use $Q$ rather than the smallest distance?

A1.

**Q2.** A branch has bootstrap support 87 and posterior probability 0.87. How was each number made, and what does each mean?

A2.

**Q3.** Score the same three-leaf tree by Fitch for the column C, T, T (left pair C, T; right leaf T). How many changes, and how many ways to place them?

A3.